<a href="https://colab.research.google.com/github/Omkar675/Hospital-Management-System/blob/main/the__bank_transfer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, auc
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import pickle
import warnings
warnings.filterwarnings('ignore')

In [2]:

print(" LOADING DATA")


df = pd.read_csv('/content/PS_20174392719_1491204439457_log.csv')
df.head()

 LOADING DATA


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
df.shape

(6362620, 11)

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [5]:
print(f"\nFraud distribution:")
print(df['isFraud'].value_counts())


Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [6]:
print(f"\nMissing values:\n{df.isnull().sum()}")


Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64


In [7]:
fraud_rate = df['isFraud'].mean() * 100
print(f"\nFraud rate: {fraud_rate:.2f}%")


Fraud rate: 0.13%


In [8]:
print(f"\nFraud by transaction type:")
print(df.groupby('type')['isFraud'].agg(['sum', 'mean']))


Fraud by transaction type:
           sum      mean
type                    
CASH_IN      0  0.000000
CASH_OUT  4116  0.001840
DEBIT        0  0.000000
PAYMENT      0  0.000000
TRANSFER  4097  0.007688


In [9]:
# Create new features
df['balanceChangeOrig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balanceChangeDest'] = df['newbalanceDest'] - df['oldbalanceDest']
df['hour'] = df['step'] % 24
df['day'] = df['step'] // 24

In [10]:
# Encode transaction type
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])

In [11]:
# Select features for modeling (matching the exact names we created above)
feature_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
                'newbalanceDest', 'balanceChangeOrig', 'balanceChangeDest',
                'hour', 'day', 'errorBalanceOrig', 'errorBalanceDest', 'type_encoded']


In [13]:
# Verify all features are created

print("FEATURE VERIFICATION")


required_features = ['balanceChangeOrig', 'balanceChangeDest', 'hour', 'day',
                     'errorBalanceOrig', 'errorBalanceDest', 'type_encoded']

print("\nChecking if all engineered features exist:")
for feature in required_features:
    if feature in df.columns:
        print(f"✓ {feature}: EXISTS")
    else:
        print(f"✗ {feature}: MISSING")

FEATURE VERIFICATION

Checking if all engineered features exist:
✓ balanceChangeOrig: EXISTS
✓ balanceChangeDest: EXISTS
✓ hour: EXISTS
✓ day: EXISTS
✗ errorBalanceOrig: MISSING
✗ errorBalanceDest: MISSING
✓ type_encoded: EXISTS


In [ ]:

# 4. TRAIN-TEST SPLIT


print("TRAIN-TEST SPLIT")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Training fraud rate: {y_train.mean() * 100:.2f}%")
print(f"Test fraud rate: {y_test.mean() * 100:.2f}%")

TRAIN-TEST SPLIT
Training set size: (5090096, 10)
Test set size: (1272524, 10)
Training fraud rate: 0.13%
Test fraud rate: 0.13%


In [ ]:
# ============================================================================
# 5. FEATURE SCALING
# ============================================================================

print(" FEATURE SCALING")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully!")

 FEATURE SCALING
Features scaled successfully!


In [ ]:
# ============================================================================
# 6. APPLY SMOTE
# ============================================================================

print(" APPLYING SMOTE")


print(f"Before SMOTE - Fraud cases: {y_train.sum()}, Normal cases: {(y_train == 0).sum()}")

smote = SMOTE(random_state=42, sampling_strategy=0.5)  # 0.5 means minority will be 50% of majority
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE - Fraud cases: {y_train_smote.sum()}, Normal cases: {(y_train_smote == 0).sum()}")


 APPLYING SMOTE
Before SMOTE - Fraud cases: 6570, Normal cases: 5083526
After SMOTE - Fraud cases: 2541763, Normal cases: 5083526


In [ ]:
# 7. MODEL TRAINING
# ============================================================================

print(" TRAINING MODELS")


models = {}
results = {}


 TRAINING MODELS


In [ ]:
# Train model
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_smote, y_train_smote)
print("\n✓ Model trained successfully!")



✓ Model trained successfully!


In [ ]:
# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_lr))



CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      0.96      0.98   1270881
           1       0.03      0.89      0.05      1643

    accuracy                           0.96   1272524
   macro avg       0.51      0.92      0.52   1272524
weighted avg       1.00      0.96      0.98   1272524



In [ ]:
# Calculate metrics
roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_pred_proba_lr)
pr_auc_lr = auc(recall_lr, precision_lr)

print(f"\nROC-AUC Score: {roc_auc_lr:.4f}")
print(f"PR-AUC Score: {pr_auc_lr:.4f}")


ROC-AUC Score: 0.9774
PR-AUC Score: 0.5665


In [ ]:
# random forest

# Train model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf_model.fit(X_train_smote, y_train_smote)
print("\n✓ Model trained successfully!")

# Predictions
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_rf))


✓ Model trained successfully!

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.52      0.98      0.68      1643

    accuracy                           1.00   1272524
   macro avg       0.76      0.99      0.84   1272524
weighted avg       1.00      1.00      1.00   1272524



In [ ]:
# XGBOOST

scale_pos_weight = (y_train_smote == 0).sum() / y_train_smote.sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train_smote, y_train_smote)
print("\n✓ Model trained successfully!")

# Predictions
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_xgb))


✓ Model trained successfully!

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1270881
           1       0.19      0.99      0.32      1643

    accuracy                           0.99   1272524
   macro avg       0.60      0.99      0.66   1272524
weighted avg       1.00      0.99      1.00   1272524

